# Phase 4 — Raw EEG Deep Learning

I inspect the input-output behavior of a 1D-CNN, CNN-BiLSTM and lightweight Transformer before training.

> **Research integrity:** Figures labeled synthetic are not clinical EEG observations. Run code locally in VS Code using the project `.venv` interpreter.

## 1. Three architectures

- CNN learns local temporal filters.
- CNN-BiLSTM incorporates sequence dynamics after convolutional reduction.
- Transformer learns contextual relationships between learned time tokens.

All are compact educational baselines; no state-of-the-art claim is made.

In [ ]:
import torch
from eeg_sleep.models import get_model
x=torch.randn(2,2,3000)
for name in ("cnn","lstm","transformer"):
    model=get_model(name,channels=2,classes=5)
    with torch.no_grad(): logits=model(x)
    print(name,"logits:",tuple(logits.shape),"parameters:",sum(p.numel() for p in model.parameters()))

## 2. Softmax converts logits into predicted class probabilities

In [ ]:
model=get_model("cnn")
model.eval()
with torch.no_grad():
    probabilities=model(x).softmax(dim=1)
print("Shape:",tuple(probabilities.shape))
print("Row sums:",probabilities.sum(dim=1))

## 3. Train normalization is derived from training subjects

The deep-learning pipeline computes a channel mean and standard deviation from training subjects only. Validation and test subjects are transformed using those fixed parameters.

In [ ]:
from pathlib import Path
from eeg_sleep.synthetic import make_demo_dataset
from eeg_sleep.dataset import load_arrays
from eeg_sleep.splits import subject_split
from eeg_sleep.deep_learning import normalize_by_train
p=Path("data/processed/synthetic_demo.npz")
if not p.exists():make_demo_dataset(p,n_subjects=10,repeats_per_stage=2)
X,y,groups,ids,fs=load_arrays(p)
tr,va,te=subject_split(groups)
Z,mean,std=normalize_by_train(X,tr)
print("Input shape:",X.shape,"Normalized shape:",Z.shape)
print("Training-only means:",mean.squeeze())

## 4. Optional local training

Change `RUN_TRAINING` to `True` only when ready. Real-data training needs a separate prepared Sleep-EDF `.npz`. No background job is started.

In [ ]:
RUN_TRAINING=False
if RUN_TRAINING:
    from eeg_sleep.deep_learning import train_deep
    for architecture in ("cnn", "lstm", "transformer"):
        metrics=train_deep(p, f"results/demo_{architecture}",architecture=architecture,
                           epochs=2,batch_size=16,device_name="auto")
        print(architecture,"synthetic macro F1:",metrics["macro_f1"])
else: print("Training disabled; architectures and tensor shapes already verified.")

## Exercises

1. Explain Conv1d input `(batch, channels, time)`. 2. Why should normalization be fit on train only? 3. What does early stopping monitor? 4. Compare the computational costs and assumptions of the models. 5. Why are synthetic metrics not biomedical findings?